# Week 3b: serving experiments on the T4s

**Kaggle settings:** Accelerator = *GPU T4 x2*, Internet = *on*.

`python scripts/kaggle_run.py run --week 3 --part b --kernel-source qwen-sql-week3a`

Mounts week3a's output (fp16 + AWQ + GPTQ + W8A8 checkpoints, quality results). Runs every experiment in `configs/experiments.yaml` (one flag changed vs the fp16 baseline, which is re-measured first in this session), checks fp8 KV-cache accuracy, then builds the experiment matrix and Pareto plot. Finished experiments are skipped on re-run.

In [ ]:
BASE = "Qwen/Qwen3-1.7B"
OUT = "/kaggle/working/week3b"
SPIDER = "/kaggle/working/data/spider_data"
WL = f"{OUT}/workloads"

import os, subprocess, json, glob, shutil
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for k in ["MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"]:
    try:
        os.environ[k] = _s.get_secret(k)
    except Exception:
        print(f"secret {k} not set; that integration stays off")
os.makedirs(OUT, exist_ok=True)
STATUS = {}

def run(step, cmd, needs=()):
    """Run a shell step, stream its output, keep a per-step log, never raise.
    A failed step marks itself so that steps depending on it are skipped, not re-failed."""
    blocked = [n for n in needs if STATUS.get(n) != "ok"]
    if blocked:
        STATUS[step] = "skipped"
        print(f"--- skip {step}: needs {blocked}")
        return False
    print(f"--- {step}: {cmd}")
    with open(f"{OUT}/{step}.log", "w") as log:
        proc = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in proc.stdout:
            log.write(line)
            if "it/s]" not in line and "s/it]" not in line:
                print(line, end="")
    STATUS[step] = "ok" if proc.wait() == 0 else "failed"
    print(f"--- {step}: {STATUS[step]}")
    return STATUS[step] == "ok"

In [ ]:
import glob, shutil, subprocess, tarfile
DEST = "/kaggle/working/repo"
marker = glob.glob("/kaggle/input/**/COMMIT", recursive=True)
if marker:  # code uploaded by scripts/kaggle_run.py
    commit = open(marker[0]).read().strip()
    tars = glob.glob("/kaggle/input/**/src.tar", recursive=True)
    trees = glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
    os.makedirs(DEST, exist_ok=True)
    if tars:
        with tarfile.open(tars[0]) as t:
            t.extractall(DEST, filter="data")
    elif trees:  # Kaggle unpacks uploaded archives itself (into src/)
        shutil.copytree(os.path.dirname(trees[0]), DEST, dirs_exist_ok=True)
    else:
        raise RuntimeError("code dataset attached but neither src.tar nor pyproject.toml found")
else:
    !git clone -q {REPO} {DEST}
    commit = subprocess.getoutput(f"git -C {DEST} rev-parse --short HEAD")
open(f"{OUT}/COMMIT", "w").write(commit + "\n")
print("code commit:", commit)
%cd {DEST}
!pip install -q -e . -r requirements-gpu.txt 2>&1 | tail -3
# The image ships torchao 0.10; PEFT 0.20 raises on torchao < 0.16 when loading LoRA onto a plain
# (non-quantized) model, which broke the merge in run 3. Nothing here uses torchao.
!pip uninstall -y -q torchao

# Fail fast: a broken setup must not burn GPU hours on cells that can only fail.
check = subprocess.run(["python", "-c", "import bench.experiments, bench.matrix, bench.client, evaluation.generate, vllm"],
                       capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("repo or serving deps not importable:\n" + check.stderr[-2000:])
print("repo and serving deps import OK")

In [ ]:
manifest = glob.glob("/kaggle/input/**/models/MANIFEST.json", recursive=True)
if not manifest:
    raise RuntimeError("week3a output not mounted: run with --kernel-source qwen-sql-week3a")
MODELS = os.path.dirname(manifest[0])
A_OUT = os.path.join(os.path.dirname(MODELS), "week3a")  # week3a's quality results
print("models:", json.load(open(manifest[0])), "\nweek3a results:", A_OUT, os.path.isdir(A_OUT))

## 1. Environment record

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv | tee {OUT}/gpu.txt
!pip freeze | grep -iE "^(vllm|torch|transformers|peft|llmcompressor|compressed-tensors)==" | tee {OUT}/versions.txt
!nproc | tee {OUT}/cpus.txt

## 2. Data and frozen workloads (same tokenizer and seed as Week 2: MANIFEST.json hashes must match for the v1 workloads)

In [ ]:
!python scripts/get_spider.py --out /kaggle/working/data
run("workloads", f"python -m bench.workloads --tokenizer {BASE} --out {WL} --spider-dir {SPIDER}")

## 3. Does fp8 KV cache cost accuracy?
Only e5m2 is checked: Triton on sm75 has no e4m3 conversions (experiment `kv_fp8_e4m3` records whether the server even starts).

In [ ]:
run("eval_fp16_kv_fp8_e5m2", f"CUDA_VISIBLE_DEVICES=0 python -m evaluation.generate --model {MODELS}/fp16 --kv-cache-dtype fp8_e5m2 "
    f"--spider-dir {SPIDER} --out {OUT}/preds/fp16_kv_fp8_e5m2.jsonl")
run("compare_fp16_kv_fp8_e5m2", f"python -m evaluation.compare --spider-dir {SPIDER} --a {A_OUT}/preds/fp16.jsonl "
    f"--b {OUT}/preds/fp16_kv_fp8_e5m2.jsonl --out {OUT}/compare/fp16_kv_fp8_e5m2.json", needs=["eval_fp16_kv_fp8_e5m2"])

## 4. Serving experiments (configs/experiments.yaml)
One server config at a time; a config that fails to start on sm75 is recorded with its log tail, and the run continues.

In [ ]:
run("experiments", f"python -m bench.experiments --models-dir {MODELS} --workloads {WL} --out {OUT}/experiments",
    needs=["workloads"])
print(open(f"{OUT}/experiments/status.json").read() if os.path.exists(f"{OUT}/experiments/status.json") else "no status")

## 5. Experiment matrix + Pareto plot, results zip

In [ ]:
run("matrix", f"python -m bench.matrix --experiments {OUT}/experiments --quality {A_OUT} {OUT} --out {OUT}/report")
shutil.copy(f"{WL}/MANIFEST.json", OUT) if os.path.exists(f"{WL}/MANIFEST.json") else print("missing MANIFEST")
shutil.rmtree(WL, ignore_errors=True)  # regenerated deterministically; the manifest proves identity
json.dump(STATUS, open(f"{OUT}/status.json", "w"), indent=2)
print("step status:", STATUS)
shutil.make_archive("/kaggle/working/week3b_results", "zip", OUT)
for d in ["/kaggle/working/data"]:  # keep kernel output small
    shutil.rmtree(d, ignore_errors=True)
print("download /kaggle/working/week3b_results.zip")